In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True,
)

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "Amazon-Marketplace-Product-Intelligence-Platform-BJIT"
)

print("Project exists:", PROJECT_ROOT.exists())
print("Project path:", PROJECT_ROOT)

Project exists: True
Project path: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT


## Step 3 — Build and compare recommendation approaches

What we are doing now:

Build TF-IDF similarity.
Generate MiniLM semantic embeddings.
Build an initial hybrid recommender.
Compare the three approaches.
Verify deterministic recommendations.
Save reusable intermediate artifacts.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd
import torch

from scipy.sparse import save_npz
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import MultiLabelBinarizer
from sentence_transformers import SentenceTransformer


drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "Amazon-Marketplace-Product-Intelligence-Platform-BJIT"
)

CLEAN_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "products_recommendation_clean.csv"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "recommendation"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "recommendation"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

products = pd.read_csv(CLEAN_DATA_PATH)

products["asin"] = products["asin"].astype(str)

for column in [
    "title",
    "description",
    "brand",
    "keyword_list",
]:
    products[column] = (
        products[column]
        .fillna("")
        .astype(str)
    )

products = (
    products
    .sort_values("asin")
    .reset_index(drop=True)
)

print("Products loaded:", len(products))
print("Unique ASINs:", products["asin"].nunique())
print(
    "Device:",
    "cuda" if torch.cuda.is_available() else "cpu",
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Products loaded: 981
Unique ASINs: 981
Device: cuda


## Cell 3 — Prepare model-specific text

TF-IDF can process the full description. MiniLM input is kept shorter because the model has a 256-token limit.

In [ ]:
description_preview = (
    products["description"]
    .str.split()
    .str[:170]
    .str.join(" ")
)

# Keep Unknown for display but exclude it from model text.
brand_for_model = (
    products["brand"]
    .where(
        ~products["brand"]
        .str.lower()
        .isin(["", "unknown", "nan"]),
        "",
    )
)

# Production text used by the final application.
# Keyword remains a supporting recommendation feature.
products["lexical_text"] = (
    products["title"]
    + " "
    + products["title"]
    + " "
    + brand_for_model
    + " "
    + products["keyword_list"]
    + " "
    + products["keyword_list"]
    + " "
    + products["description"]
)

products["semantic_text"] = (
    products["title"]
    + ". Brand: "
    + brand_for_model
    + ". Product type: "
    + products["keyword_list"]
    + ". Description: "
    + description_preview
)

# Evaluation-only text. Keyword is excluded to prevent leakage.
products["evaluation_lexical_text"] = (
    products["title"]
    + " "
    + products["title"]
    + " "
    + brand_for_model
    + " "
    + products["description"]
)

products["evaluation_semantic_text"] = (
    products["title"]
    + ". Brand: "
    + brand_for_model
    + ". Description: "
    + description_preview
)

print("Production and evaluation text prepared.")

display(
    products[
        [
            "asin",
            "title",
            "brand",
            "keyword_list",
            "lexical_text",
            "evaluation_lexical_text",
        ]
    ].head(2)
)

Production and evaluation text prepared.


,asin,title,brand,keyword_list,lexical_text,evaluation_lexical_text
0,8883707168,"""Moleskine Classic Notebook, Soft Cover, Large...",Moleskine,notebook journal,"""Moleskine Classic Notebook, Soft Cover, Large...","""Moleskine Classic Notebook, Soft Cover, Large..."
1,B00005IBX9,Cuisinart DCC-1200P1 Brew Central 12-Cup Progr...,Cuisinart,coffee maker,Cuisinart DCC-1200P1 Brew Central 12-Cup Progr...,Cuisinart DCC-1200P1 Brew Central 12-Cup Progr...


## Cell 4 — Build TF-IDF baseline

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=25000,
    sublinear_tf=True,
    norm="l2",
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    products["lexical_text"]
)

tfidf_similarity = linear_kernel(
    tfidf_matrix,
    tfidf_matrix,
).astype("float32")

print("TF-IDF matrix:", tfidf_matrix.shape)
print(
    "Vocabulary size:",
    len(tfidf_vectorizer.vocabulary_),
)

TF-IDF matrix: (981, 25000)
Vocabulary size: 25000


## Cell 5 — Generate semantic embeddings

In [ ]:
SEMANTIC_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

semantic_model = SentenceTransformer(
    SEMANTIC_MODEL_NAME,
    device=device,
)

semantic_model.max_seq_length = 256

semantic_embeddings = semantic_model.encode(
    products["semantic_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

semantic_embeddings = semantic_embeddings.astype(
    "float32"
)

semantic_similarity = (
    semantic_embeddings
    @ semantic_embeddings.T
)

semantic_similarity = np.clip(
    semantic_similarity,
    0,
    1,
).astype("float32")

print(
    "Semantic embeddings:",
    semantic_embeddings.shape,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Semantic embeddings: (981, 384)


## Cell 6 — Create brand and keyword signals

In [ ]:
normalized_brands = (
    products["brand"]
    .str.lower()
    .str.strip()
    .to_numpy()
)

brand_signal = (
    normalized_brands[:, None]
    == normalized_brands[None, :]
).astype("float32")

unknown_brand_mask = np.isin(
    normalized_brands,
    ["", "unknown", "nan"],
)

brand_signal[
    unknown_brand_mask,
    :
] = 0

brand_signal[
    :,
    unknown_brand_mask,
] = 0


keyword_sets = (
    products["keyword_list"]
    .str.split(" | ", regex=False)
    .apply(
        lambda values: [
            value.strip()
            for value in values
            if value.strip()
        ]
    )
)

keyword_encoder = MultiLabelBinarizer()

keyword_matrix = keyword_encoder.fit_transform(
    keyword_sets
)

keyword_signal = (
    keyword_matrix
    @ keyword_matrix.T
    > 0
).astype("float32")

print("Brand signal:", brand_signal.shape)
print("Keyword signal:", keyword_signal.shape)
print(
    "Unique keywords:",
    len(keyword_encoder.classes_),
)

Brand signal: (981, 981)
Keyword signal: (981, 981)
Unique keywords: 28


## Cell 7 — Create the initial hybrid score
These are starting weights only. They will be tuned in the next step.

In [ ]:
INITIAL_WEIGHTS = {
    "semantic": 0.45,
    "tfidf": 0.35,
    "brand": 0.10,
    "keyword": 0.10,
}

hybrid_similarity = (
    INITIAL_WEIGHTS["semantic"]
    * semantic_similarity

    + INITIAL_WEIGHTS["tfidf"]
    * tfidf_similarity

    + INITIAL_WEIGHTS["brand"]
    * brand_signal

    + INITIAL_WEIGHTS["keyword"]
    * keyword_signal
).astype("float32")

similarity_matrices = {
    "TF-IDF": tfidf_similarity,
    "Semantic MiniLM": semantic_similarity,
    "Hybrid": hybrid_similarity,
}

print("Hybrid similarity created.")
print("Initial weights:", INITIAL_WEIGHTS)

Hybrid similarity created.
Initial weights: {'semantic': 0.45, 'tfidf': 0.35, 'brand': 0.1, 'keyword': 0.1}


## Cell 8 — Deterministic recommendation function

In [ ]:
asin_to_index = pd.Series(
    products.index,
    index=products["asin"],
).to_dict()

asin_values = products["asin"].to_numpy()


def get_top_indices(
    similarity_matrix,
    product_index,
    top_k=5,
):
    scores = similarity_matrix[
        product_index
    ].copy()

    # Prevent recommending the selected product itself.
    scores[product_index] = -np.inf

    # Score descending, ASIN ascending for stable ties.
    ranked_indices = np.lexsort(
        (
            asin_values,
            -scores,
        )
    )

    return ranked_indices[:top_k]


def recommend_by_asin(
    asin,
    approach="Hybrid",
    top_k=5,
):
    asin = str(asin).strip()

    if asin not in asin_to_index:
        raise ValueError(
            f"ASIN not found: {asin}"
        )

    product_index = asin_to_index[asin]
    similarity_matrix = similarity_matrices[
        approach
    ]

    result_indices = get_top_indices(
        similarity_matrix,
        product_index,
        top_k,
    )

    results = products.loc[
        result_indices,
        [
            "asin",
            "title",
            "brand",
            "keyword_list",
            "price",
            "avg_rating",
            "image_url",
            "product_url",
        ],
    ].copy()

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1),
    )

    results["similarity_score"] = (
        similarity_matrix[
            product_index,
            result_indices,
        ]
    )

    return results.reset_index(drop=True)

## Cell 9 — Compare sample recommendations

In [ ]:
SAMPLE_ASIN = products.iloc[0]["asin"]

selected_product = products[
    products["asin"].eq(SAMPLE_ASIN)
][
    [
        "asin",
        "title",
        "brand",
        "keyword_list",
    ]
]

print("Selected product")
display(selected_product)

for approach_name in similarity_matrices:
    print("\nApproach:", approach_name)

    display(
        recommend_by_asin(
            SAMPLE_ASIN,
            approach=approach_name,
            top_k=5,
        )[
            [
                "rank",
                "asin",
                "title",
                "brand",
                "keyword_list",
                "similarity_score",
            ]
        ]
    )

Selected product


,asin,title,brand,keyword_list
0,8883707168,"""Moleskine Classic Notebook, Soft Cover, Large...",Moleskine,notebook journal



Approach: TF-IDF


,rank,asin,title,brand,keyword_list,similarity_score
0,1,B0F38C4WYF,"6 Pack Lined Journal Notebooks for Work, A5 Ha...",ZCZN,notebook journal,0.279150
1,2,B083QFCTTT,Nice Black Notebook Ruled - Soft Cover Faux Le...,LeStallion,notebook journal,0.277378
2,3,B0C23S36QF,"Fetinar A5 Faux Leather Journal Notebook, 300 ...",Fetinar,notebook journal,0.246741
3,4,B08YNMTT2D,"RETTACY Lined Journal Notebook, 192 Numbered P...",RETTACY,notebook journal,0.233045
4,5,B0C1Z48WKW,"feela 8 Pack Soft Cover Journal Bulk, Notebook...",feela,notebook journal,0.218851



Approach: Semantic MiniLM


,rank,asin,title,brand,keyword_list,similarity_score
0,1,B01DN8TCEU,Amazon Basics Classic Lined Writing Notebook f...,Amazon Basics,notebook journal,0.710233
1,2,B0GVSM61BG,"8 Pack Lined Journal Notebook for Work, A5 Har...",BGUP,notebook journal,0.679293
2,3,B0F38C4WYF,"6 Pack Lined Journal Notebooks for Work, A5 Ha...",ZCZN,notebook journal,0.664869
3,4,B0GTDBXDZT,"Lined Journal Notebook, A5 Hardcover Leather J...",KWULP,notebook journal,0.660140
4,5,B0C8JZ57R1,"Hardcover Lined Notebook Journal for Writing, ...",FOCUS DAY,notebook journal,0.657288



Approach: Hybrid


,rank,asin,title,brand,keyword_list,similarity_score
0,1,B0F38C4WYF,"6 Pack Lined Journal Notebooks for Work, A5 Ha...",ZCZN,notebook journal,0.496894
1,2,B01DN8TCEU,Amazon Basics Classic Lined Writing Notebook f...,Amazon Basics,notebook journal,0.492786
2,3,B083QFCTTT,Nice Black Notebook Ruled - Soft Cover Faux Le...,LeStallion,notebook journal,0.471942
3,4,B0GVSM61BG,"8 Pack Lined Journal Notebook for Work, A5 Har...",BGUP,notebook journal,0.468391
4,5,B08YNMTT2D,"RETTACY Lined Journal Notebook, 192 Numbered P...",RETTACY,notebook journal,0.467062


# Cell 10 — Evaluate the three approaches

In [ ]:
def evaluate_recommender(
    similarity_matrix,
    top_k=5,
):
    precision_scores = []
    hit_scores = []
    recommended_products = set()
    evaluated_queries = 0

    for product_index in range(len(products)):
        relevant_mask = (
            keyword_signal[product_index]
            .astype(bool)
        )

        relevant_mask[product_index] = False

        if not relevant_mask.any():
            continue

        top_indices = get_top_indices(
            similarity_matrix,
            product_index,
            top_k,
        )

        precision_scores.append(
            relevant_mask[
                top_indices
            ].mean()
        )

        hit_scores.append(
            relevant_mask[
                top_indices
            ].any()
        )

        recommended_products.update(
            top_indices.tolist()
        )

        evaluated_queries += 1

    return {
        "evaluated_queries": evaluated_queries,
        f"keyword_precision@{top_k}":
            np.mean(precision_scores),
        f"keyword_hit_rate@{top_k}":
            np.mean(hit_scores),
        "catalog_coverage":
            len(recommended_products)
            / len(products),
    }


comparison_rows = []

for approach_name, matrix in (
    similarity_matrices.items()
):
    metrics = evaluate_recommender(
        matrix,
        top_k=5,
    )

    comparison_rows.append({
        "approach": approach_name,
        **metrics,
    })

comparison_results = pd.DataFrame(
    comparison_rows
).sort_values(
    "keyword_precision@5",
    ascending=False,
).reset_index(drop=True)

display(
    comparison_results.style.format({
        "keyword_precision@5": "{:.4f}",
        "keyword_hit_rate@5": "{:.4f}",
        "catalog_coverage": "{:.4f}",
    })
)

,approach,evaluated_queries,keyword_precision@5,keyword_hit_rate@5,catalog_coverage
0,Hybrid,981,0.9973,1.0000,0.9439
1,TF-IDF,981,0.9945,1.0000,0.9623
2,Semantic MiniLM,981,0.9896,0.9980,0.9042


## Cell 11 — Verify deterministic results

In [ ]:
first_result = recommend_by_asin(
    SAMPLE_ASIN,
    approach="Hybrid",
    top_k=10,
)

second_result = recommend_by_asin(
    SAMPLE_ASIN,
    approach="Hybrid",
    top_k=10,
)

deterministic_result = (
    first_result["asin"].tolist()
    == second_result["asin"].tolist()
)

print(
    "Same recommendations for same input:",
    deterministic_result,
)

Same recommendations for same input: True


## Cell 12 — Save reusable artifacts

In [ ]:
joblib.dump(
    tfidf_vectorizer,
    MODEL_DIR / "tfidf_vectorizer.joblib",
)

save_npz(
    MODEL_DIR / "tfidf_matrix.npz",
    tfidf_matrix,
)

np.save(
    MODEL_DIR / "semantic_embeddings.npy",
    semantic_embeddings,
)

PRODUCT_INDEX_PATH = (
    MODEL_DIR
    / "recommendation_product_index.csv"
)

products.to_csv(
    PRODUCT_INDEX_PATH,
    index=False,
    encoding="utf-8-sig",
)

COMPARISON_PATH = (
    REPORT_DIR
    / "recommendation_approach_comparison.csv"
)

comparison_results.to_csv(
    COMPARISON_PATH,
    index=False,
    encoding="utf-8-sig",
)

development_metadata = {
    "semantic_model": SEMANTIC_MODEL_NAME,
    "product_count": len(products),
    "embedding_dimension": int(
        semantic_embeddings.shape[1]
    ),
    "tfidf_features": int(
        tfidf_matrix.shape[1]
    ),
    "initial_weights": INITIAL_WEIGHTS,
    "deterministic": bool(
        deterministic_result
    ),
}

with open(
    MODEL_DIR / "development_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        development_metadata,
        file,
        indent=2,
    )

print("Intermediate artifacts saved successfully.")
print("Model directory:", MODEL_DIR)
print("Comparison report:", COMPARISON_PATH)

Intermediate artifacts saved successfully.
Model directory: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/models/recommendation
Comparison report: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/recommendation/recommendation_approach_comparison.csv


## Step 4A — Build Leakage-Free Evaluation Representations

Create separate TF-IDF and semantic representations from product title, brand, and description only. Collection keywords are excluded to prevent target leakage during automatic evaluation.

In [ ]:
# ============================================================
# Step 4A: Build leakage-free evaluation representations
# ============================================================

required_columns = [
    "evaluation_lexical_text",
    "evaluation_semantic_text",
]

missing_columns = [
    column
    for column in required_columns
    if column not in products.columns
]

if missing_columns:
    raise ValueError(
        f"Missing evaluation columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 1. Evaluation-only TF-IDF representation
# ------------------------------------------------------------

evaluation_tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=25000,
    sublinear_tf=True,
    norm="l2",
)

evaluation_tfidf_matrix = (
    evaluation_tfidf_vectorizer.fit_transform(
        products["evaluation_lexical_text"]
    )
)

evaluation_tfidf_similarity = linear_kernel(
    evaluation_tfidf_matrix,
    evaluation_tfidf_matrix,
).astype("float32")


# ------------------------------------------------------------
# 2. Evaluation-only MiniLM representation
# ------------------------------------------------------------

evaluation_semantic_embeddings = (
    semantic_model.encode(
        products["evaluation_semantic_text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    .astype("float32")
)

evaluation_semantic_similarity = (
    evaluation_semantic_embeddings
    @ evaluation_semantic_embeddings.T
)

evaluation_semantic_similarity = np.clip(
    evaluation_semantic_similarity,
    0,
    1,
).astype("float32")


# ------------------------------------------------------------
# 3. Validation checks
# ------------------------------------------------------------

expected_rows = len(products)

assert evaluation_tfidf_matrix.shape[0] == expected_rows
assert evaluation_semantic_embeddings.shape[0] == expected_rows
assert evaluation_tfidf_similarity.shape == (
    expected_rows,
    expected_rows,
)
assert evaluation_semantic_similarity.shape == (
    expected_rows,
    expected_rows,
)

print("Leakage-free evaluation representations created.")
print(
    "Evaluation TF-IDF matrix:",
    evaluation_tfidf_matrix.shape,
)
print(
    "Evaluation TF-IDF vocabulary:",
    len(evaluation_tfidf_vectorizer.vocabulary_),
)
print(
    "Evaluation semantic embeddings:",
    evaluation_semantic_embeddings.shape,
)
print(
    "Evaluation TF-IDF similarity:",
    evaluation_tfidf_similarity.shape,
)
print(
    "Evaluation semantic similarity:",
    evaluation_semantic_similarity.shape,
)

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Leakage-free evaluation representations created.
Evaluation TF-IDF matrix: (981, 25000)
Evaluation TF-IDF vocabulary: 25000
Evaluation semantic embeddings: (981, 384)
Evaluation TF-IDF similarity: (981, 981)
Evaluation semantic similarity: (981, 981)


## Step 4B — Prepare Automatic Relevance Labels

Collection keywords are used as weak relevance labels. Two products are considered relevant when they share at least one collection keyword. These labels are used only for evaluation and are excluded from the evaluation input text.

In [ ]:
# ============================================================
# Step 4B: Prepare automatic relevance labels
# ============================================================

EVALUATION_TOP_K = 5

# Convert the existing keyword-overlap signal into
# a Boolean relevance matrix.
evaluation_relevance_matrix = (
    keyword_signal
    .astype(bool)
    .copy()
)

# A product must never be relevant to itself.
np.fill_diagonal(
    evaluation_relevance_matrix,
    False,
)

# Number of relevant alternatives available for each product.
evaluation_relevant_counts = (
    evaluation_relevance_matrix.sum(axis=1)
)

# Only evaluate products that have at least one
# relevant alternative.
evaluation_query_indices = np.flatnonzero(
    evaluation_relevant_counts > 0
)

evaluation_query_table = products.loc[
    evaluation_query_indices,
    [
        "asin",
        "title",
        "brand",
        "keyword_list",
    ],
].copy()

evaluation_query_table.insert(
    0,
    "product_index",
    evaluation_query_indices,
)

evaluation_query_table["relevant_product_count"] = (
    evaluation_relevant_counts[
        evaluation_query_indices
    ]
)


# ------------------------------------------------------------
# Keyword distribution summary
# ------------------------------------------------------------

keyword_frequency = pd.Series(
    [
        keyword
        for product_keywords in keyword_sets
        for keyword in product_keywords
    ],
    name="keyword",
).value_counts()

keyword_distribution = (
    keyword_frequency
    .rename_axis("keyword")
    .reset_index(name="product_count")
)


# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert (
    evaluation_relevance_matrix.shape
    == (len(products), len(products))
)

assert not np.diag(
    evaluation_relevance_matrix
).any()

assert len(evaluation_query_indices) > 0

print("Automatic relevance labels prepared.")
print("Total products:", len(products))
print(
    "Evaluation queries:",
    len(evaluation_query_indices),
)
print(
    "Queries without relevant alternatives:",
    int(
        (
            evaluation_relevant_counts == 0
        ).sum()
    ),
)
print(
    "Unique collection keywords:",
    len(keyword_distribution),
)
print(
    "Minimum relevant products per query:",
    int(
        evaluation_relevant_counts[
            evaluation_query_indices
        ].min()
    ),
)
print(
    "Maximum relevant products per query:",
    int(
        evaluation_relevant_counts[
            evaluation_query_indices
        ].max()
    ),
)

display(keyword_distribution)
display(evaluation_query_table.head())

Automatic relevance labels prepared.
Total products: 981
Evaluation queries: 981
Queries without relevant alternatives: 0
Unique collection keywords: 28
Minimum relevant products per query: 15
Maximum relevant products per query: 59


,keyword,product_count
0,sneakers,60
1,coffee maker,58
2,water bottle,58
3,sunglasses,57
4,t-shirt,57
5,yamaha accoustic guitar,56
6,notebook journal,55
7,gaming chair,55
8,kitchen knife set,54
9,electric toothbrush,54


,product_index,asin,title,brand,keyword_list,relevant_product_count
0,0,8883707168,"""Moleskine Classic Notebook, Soft Cover, Large...",Moleskine,notebook journal,54
1,1,B00005IBX9,Cuisinart DCC-1200P1 Brew Central 12-Cup Progr...,Cuisinart,coffee maker,57
2,2,B00006IV0Q,Presto 02811 Stainless Steel 12-Cup Electric C...,Presto,coffee maker,57
3,3,B00009J580,"Yamaha JR1 FG Junior 3/4 Size Acoustic Guitar,...",Yamaha,yamaha accoustic guitar,55
4,4,B0006ZAU22,Washburn Deep Forest Burl ACE Acoustic Electri...,Washburn,yamaha accoustic guitar,55


# Step 4C — Evaluate TF-IDF and MiniLM

This step compares only the leakage-free TF-IDF and MiniLM representations. Keywords are used only to score their recommendations.

In [ ]:
# ============================================================
# Step 4C: Evaluate TF-IDF and MiniLM
# ============================================================

from time import perf_counter


evaluation_similarity_matrices = {
    "TF-IDF": evaluation_tfidf_similarity,
    "MiniLM": evaluation_semantic_similarity,
}


def calculate_ndcg_at_k(
    relevance_values,
    total_relevant,
    top_k,
):
    relevance_values = np.asarray(
        relevance_values,
        dtype="float32",
    )

    discounts = 1 / np.log2(
        np.arange(2, len(relevance_values) + 2)
    )

    dcg = np.sum(
        relevance_values * discounts
    )

    ideal_result_count = min(
        int(total_relevant),
        top_k,
    )

    if ideal_result_count == 0:
        return 0.0

    ideal_dcg = np.sum(
        discounts[:ideal_result_count]
    )

    return float(dcg / ideal_dcg)


def evaluate_similarity_method(
    method_name,
    similarity_matrix,
    top_k=5,
):
    precision_scores = []
    ndcg_scores = []
    hit_scores = []
    recommended_indices = set()

    start_time = perf_counter()

    for product_index in evaluation_query_indices:

        top_indices = get_top_indices(
            similarity_matrix,
            product_index,
            top_k,
        )

        result_relevance = (
            evaluation_relevance_matrix[
                product_index,
                top_indices,
            ]
        )

        precision_scores.append(
            float(result_relevance.mean())
        )

        hit_scores.append(
            float(result_relevance.any())
        )

        ndcg_scores.append(
            calculate_ndcg_at_k(
                relevance_values=result_relevance,
                total_relevant=(
                    evaluation_relevant_counts[
                        product_index
                    ]
                ),
                top_k=top_k,
            )
        )

        recommended_indices.update(
            top_indices.tolist()
        )

    elapsed_time = (
        perf_counter() - start_time
    )

    # Confirm that ranking is repeatable.
    deterministic = all(
        np.array_equal(
            get_top_indices(
                similarity_matrix,
                product_index,
                top_k,
            ),
            get_top_indices(
                similarity_matrix,
                product_index,
                top_k,
            ),
        )
        for product_index
        in evaluation_query_indices
    )

    query_count = len(
        evaluation_query_indices
    )

    return {
        "method": method_name,
        f"ndcg@{top_k}": np.mean(
            ndcg_scores
        ),
        f"precision@{top_k}": np.mean(
            precision_scores
        ),
        f"hit_rate@{top_k}": np.mean(
            hit_scores
        ),
        "catalog_coverage": (
            len(recommended_indices)
            / len(products)
        ),
        "average_response_time_ms": (
            elapsed_time
            / query_count
            * 1000
        ),
        "deterministic": deterministic,
        "evaluated_queries": query_count,
    }


evaluation_rows = []

for method_name, similarity_matrix in (
    evaluation_similarity_matrices.items()
):
    print(f"Evaluating: {method_name}")

    evaluation_rows.append(
        evaluate_similarity_method(
            method_name=method_name,
            similarity_matrix=similarity_matrix,
            top_k=EVALUATION_TOP_K,
        )
    )


base_evaluation_results = (
    pd.DataFrame(evaluation_rows)
    .sort_values(
        f"ndcg@{EVALUATION_TOP_K}",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    base_evaluation_results.style.format({
        f"ndcg@{EVALUATION_TOP_K}": "{:.4f}",
        f"precision@{EVALUATION_TOP_K}": "{:.4f}",
        f"hit_rate@{EVALUATION_TOP_K}": "{:.4f}",
        "catalog_coverage": "{:.4f}",
        "average_response_time_ms": "{:.4f}",
    })
)

Evaluating: TF-IDF
Evaluating: MiniLM


,method,ndcg@5,precision@5,hit_rate@5,catalog_coverage,average_response_time_ms,deterministic,evaluated_queries
0,TF-IDF,0.9905,0.9900,1.0000,0.9592,0.1735,True,981
1,MiniLM,0.9883,0.9861,0.9980,0.9021,0.2264,True,981


## Step 4D — Tune the Leakage-Free Hybrid Method

Split evaluation queries into tuning and final holdout sets. Tune TF-IDF, MiniLM, and brand-similarity weights using only the tuning queries. Collection-keyword similarity is not included in the recommendation score.

In [ ]:
# ============================================================
# Step 4D: Tune the leakage-free Hybrid method
# ============================================================

from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 1. Create deterministic tuning and holdout query sets
# ------------------------------------------------------------

primary_keywords = np.array([
    product_keywords[0]
    if product_keywords
    else "unknown"
    for product_keywords in keyword_sets
])

(
    tuning_query_indices,
    final_query_indices,
) = train_test_split(
    evaluation_query_indices,
    test_size=0.30,
    random_state=42,
    stratify=primary_keywords[
        evaluation_query_indices
    ],
)

tuning_query_indices = np.sort(
    tuning_query_indices
)

final_query_indices = np.sort(
    final_query_indices
)

assert set(tuning_query_indices).isdisjoint(
    set(final_query_indices)
)

assert (
    len(tuning_query_indices)
    + len(final_query_indices)
    == len(evaluation_query_indices)
)

print("Tuning queries:", len(tuning_query_indices))
print("Final holdout queries:", len(final_query_indices))


# ------------------------------------------------------------
# 2. Evaluation function used during weight tuning
# ------------------------------------------------------------

def evaluate_tuning_quality(
    similarity_matrix,
    query_indices,
    top_k=5,
):
    precision_scores = []
    ndcg_scores = []
    hit_scores = []
    recommended_products = set()

    for product_index in query_indices:

        top_indices = get_top_indices(
            similarity_matrix,
            product_index,
            top_k,
        )

        result_relevance = (
            evaluation_relevance_matrix[
                product_index,
                top_indices,
            ]
        )

        precision_scores.append(
            float(result_relevance.mean())
        )

        hit_scores.append(
            float(result_relevance.any())
        )

        ndcg_scores.append(
            calculate_ndcg_at_k(
                relevance_values=result_relevance,
                total_relevant=(
                    evaluation_relevant_counts[
                        product_index
                    ]
                ),
                top_k=top_k,
            )
        )

        recommended_products.update(
            top_indices.tolist()
        )

    return {
        f"ndcg@{top_k}": np.mean(
            ndcg_scores
        ),
        f"precision@{top_k}": np.mean(
            precision_scores
        ),
        f"hit_rate@{top_k}": np.mean(
            hit_scores
        ),
        "catalog_coverage": (
            len(recommended_products)
            / len(products)
        ),
    }


# ------------------------------------------------------------
# 3. Tune Hybrid weights
# ------------------------------------------------------------

semantic_weight_options = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
]

brand_weight_options = [
    0.00,
    0.05,
    0.10,
]

hybrid_tuning_rows = []

for semantic_weight in semantic_weight_options:
    for brand_weight in brand_weight_options:

        tfidf_weight = (
            1.0
            - semantic_weight
            - brand_weight
        )

        if tfidf_weight <= 0:
            continue

        # Keyword signal is intentionally excluded.
        candidate_similarity = (
            tfidf_weight
            * evaluation_tfidf_similarity
            + semantic_weight
            * evaluation_semantic_similarity
            + brand_weight
            * brand_signal
        ).astype("float32")

        metrics = evaluate_tuning_quality(
            similarity_matrix=candidate_similarity,
            query_indices=tuning_query_indices,
            top_k=EVALUATION_TOP_K,
        )

        tuning_score = (
            0.60
            * metrics[f"ndcg@{EVALUATION_TOP_K}"]
            + 0.25
            * metrics[
                f"precision@{EVALUATION_TOP_K}"
            ]
            + 0.15
            * metrics["catalog_coverage"]
        )

        hybrid_tuning_rows.append({
            "tfidf_weight": tfidf_weight,
            "semantic_weight": semantic_weight,
            "brand_weight": brand_weight,
            **metrics,
            "tuning_selection_score": tuning_score,
        })


hybrid_tuning_results = (
    pd.DataFrame(hybrid_tuning_rows)
    .sort_values(
        [
            "tuning_selection_score",
            f"ndcg@{EVALUATION_TOP_K}",
            f"precision@{EVALUATION_TOP_K}",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Keep the best Hybrid configuration
# ------------------------------------------------------------

best_hybrid_row = (
    hybrid_tuning_results.iloc[0]
)

best_hybrid_weights = {
    "tfidf": float(
        best_hybrid_row["tfidf_weight"]
    ),
    "semantic": float(
        best_hybrid_row["semantic_weight"]
    ),
    "brand": float(
        best_hybrid_row["brand_weight"]
    ),
}

best_hybrid_evaluation_similarity = (
    best_hybrid_weights["tfidf"]
    * evaluation_tfidf_similarity
    + best_hybrid_weights["semantic"]
    * evaluation_semantic_similarity
    + best_hybrid_weights["brand"]
    * brand_signal
).astype("float32")


print("\nBest leakage-free Hybrid weights:")
print(best_hybrid_weights)

display(
    hybrid_tuning_results.head(10).style.format({
        "tfidf_weight": "{:.2f}",
        "semantic_weight": "{:.2f}",
        "brand_weight": "{:.2f}",
        f"ndcg@{EVALUATION_TOP_K}": "{:.4f}",
        f"precision@{EVALUATION_TOP_K}": "{:.4f}",
        f"hit_rate@{EVALUATION_TOP_K}": "{:.4f}",
        "catalog_coverage": "{:.4f}",
        "tuning_selection_score": "{:.4f}",
    })
)

Tuning queries: 686
Final holdout queries: 295

Best leakage-free Hybrid weights:
{'tfidf': 0.9, 'semantic': 0.1, 'brand': 0.0}


,tfidf_weight,semantic_weight,brand_weight,ndcg@5,precision@5,hit_rate@5,catalog_coverage,tuning_selection_score
0,0.90,0.10,0.00,0.9900,0.9898,1.0000,0.9093,0.9779
1,0.80,0.20,0.00,0.9912,0.9913,1.0000,0.9021,0.9778
2,0.60,0.40,0.00,0.9920,0.9915,1.0000,0.8940,0.9772
3,0.70,0.30,0.00,0.9914,0.9913,1.0000,0.8960,0.9771
4,0.50,0.50,0.00,0.9920,0.9913,1.0000,0.8889,0.9763
5,0.75,0.20,0.05,0.9882,0.9883,1.0000,0.9072,0.9761
6,0.65,0.30,0.05,0.9888,0.9889,1.0000,0.9001,0.9755
7,0.85,0.10,0.05,0.9868,0.9866,0.9985,0.9103,0.9753
8,0.55,0.40,0.05,0.9893,0.9886,1.0000,0.8960,0.9751
9,0.45,0.50,0.05,0.9900,0.9892,1.0000,0.8879,0.9745


## Step 4E — Final Holdout Evaluation and Method Selection

Compare TF-IDF, MiniLM, and the tuned leakage-free Hybrid on the untouched final query set. Select the final recommendation method using NDCG@5, Precision@5, and catalogue coverage.

In [ ]:
# ============================================================
# Step 4E: Final holdout evaluation and method selection
# ============================================================

def evaluate_final_method(
    method_name,
    similarity_matrix,
    query_indices,
    top_k=5,
):
    precision_scores = []
    ndcg_scores = []
    hit_scores = []
    recommended_products = set()

    start_time = perf_counter()

    for product_index in query_indices:

        top_indices = get_top_indices(
            similarity_matrix,
            product_index,
            top_k,
        )

        result_relevance = (
            evaluation_relevance_matrix[
                product_index,
                top_indices,
            ]
        )

        precision_scores.append(
            float(result_relevance.mean())
        )

        hit_scores.append(
            float(result_relevance.any())
        )

        ndcg_scores.append(
            calculate_ndcg_at_k(
                relevance_values=result_relevance,
                total_relevant=(
                    evaluation_relevant_counts[
                        product_index
                    ]
                ),
                top_k=top_k,
            )
        )

        recommended_products.update(
            top_indices.tolist()
        )

    elapsed_time = (
        perf_counter() - start_time
    )

    deterministic = all(
        np.array_equal(
            get_top_indices(
                similarity_matrix,
                product_index,
                top_k,
            ),
            get_top_indices(
                similarity_matrix,
                product_index,
                top_k,
            ),
        )
        for product_index in query_indices
    )

    query_count = len(query_indices)

    ndcg_score = float(
        np.mean(ndcg_scores)
    )

    precision_score = float(
        np.mean(precision_scores)
    )

    hit_rate = float(
        np.mean(hit_scores)
    )

    catalog_coverage = (
        len(recommended_products)
        / len(products)
    )

    selection_score = (
        0.60 * ndcg_score
        + 0.25 * precision_score
        + 0.15 * catalog_coverage
    )

    return {
        "method": method_name,
        f"ndcg@{top_k}": ndcg_score,
        f"precision@{top_k}": precision_score,
        f"hit_rate@{top_k}": hit_rate,
        "catalog_coverage": catalog_coverage,
        "average_response_time_ms": (
            elapsed_time
            / query_count
            * 1000
        ),
        "deterministic": deterministic,
        "evaluated_queries": query_count,
        "selection_score": selection_score,
    }


# ------------------------------------------------------------
# Final candidate methods
# ------------------------------------------------------------

final_candidate_matrices = {
    "TF-IDF": evaluation_tfidf_similarity,
    "MiniLM": evaluation_semantic_similarity,
    "Tuned Hybrid": (
        best_hybrid_evaluation_similarity
    ),
}


# ------------------------------------------------------------
# Evaluate only on the untouched holdout queries
# ------------------------------------------------------------

final_evaluation_rows = []

for method_name, similarity_matrix in (
    final_candidate_matrices.items()
):
    print(f"Final evaluation: {method_name}")

    final_evaluation_rows.append(
        evaluate_final_method(
            method_name=method_name,
            similarity_matrix=similarity_matrix,
            query_indices=final_query_indices,
            top_k=EVALUATION_TOP_K,
        )
    )


final_evaluation_results = (
    pd.DataFrame(final_evaluation_rows)
    .sort_values(
        [
            "selection_score",
            f"ndcg@{EVALUATION_TOP_K}",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Select final method
# ------------------------------------------------------------

selected_recommendation_method = (
    final_evaluation_results.loc[
        0,
        "method",
    ]
)

selected_evaluation_similarity = (
    final_candidate_matrices[
        selected_recommendation_method
    ]
)

assert final_evaluation_results[
    "deterministic"
].all()

assert (
    final_evaluation_results[
        "evaluated_queries"
    ]
    == len(final_query_indices)
).all()


print("\nFinal selected method:")
print(selected_recommendation_method)

if (
    selected_recommendation_method
    == "Tuned Hybrid"
):
    print(
        "Selected Hybrid weights:",
        best_hybrid_weights,
    )

display(
    final_evaluation_results.style.format({
        f"ndcg@{EVALUATION_TOP_K}": "{:.4f}",
        f"precision@{EVALUATION_TOP_K}": "{:.4f}",
        f"hit_rate@{EVALUATION_TOP_K}": "{:.4f}",
        "catalog_coverage": "{:.4f}",
        "average_response_time_ms": "{:.4f}",
        "selection_score": "{:.4f}",
    })
)

Final evaluation: TF-IDF
Final evaluation: MiniLM
Final evaluation: Tuned Hybrid

Final selected method:
TF-IDF


,method,ndcg@5,precision@5,hit_rate@5,catalog_coverage,average_response_time_ms,deterministic,evaluated_queries,selection_score
0,TF-IDF,0.9942,0.9932,1.0000,0.7401,0.2888,True,295,0.9558
1,Tuned Hybrid,0.9936,0.9925,0.9966,0.7350,0.2374,True,295,0.9545
2,MiniLM,0.9872,0.9844,0.9966,0.6809,0.2211,True,295,0.9406


### Final Recommendation Method Decision

Three content-based recommendation approaches were evaluated: TF-IDF, MiniLM semantic embeddings, and a tuned Hybrid method.

To prevent target leakage, the collection keywords were excluded from the evaluation input text. They were used only as weak relevance labels for automatically judging whether recommended products belonged to the same product group.

The evaluation queries were divided into a 70% tuning set and a 30% untouched final holdout set. Hybrid weights were tuned only on the tuning set. All final candidates were then compared using the 295 holdout queries.

The final selection score was calculated as:

- 60% NDCG@5
- 25% Precision@5
- 15% Catalogue Coverage

TF-IDF achieved the highest final selection score of 0.9558. It also achieved the highest NDCG@5, Precision@5, Hit Rate@5, and catalogue coverage. Therefore, TF-IDF was selected as the final recommendation method.

MiniLM and the tuned Hybrid were retained as experimental comparisons but were not selected for application inference. TF-IDF also provides faster application startup, lower memory usage, deterministic recommendations, and does not require loading a transformer model in Streamlit.

A limitation of this evaluation is that collection keywords are weak relevance labels rather than real customer relevance judgements. Therefore, the results measure how effectively each method reconstructs the original product groups. Future improvements can use click data, purchase history, or externally curated relevance labels.

## Step 5 — Save the final TF-IDF artifacts

This saves only the selected, leakage-free TF-IDF model for application use. Add and run:

In [ ]:
# ============================================================
# Step 5: Save final recommendation artifacts
# ============================================================

FINAL_VECTORIZER_PATH = (
    MODEL_DIR
    / "final_tfidf_vectorizer.joblib"
)

FINAL_MATRIX_PATH = (
    MODEL_DIR
    / "final_tfidf_matrix.npz"
)

FINAL_PRODUCT_INDEX_PATH = (
    MODEL_DIR
    / "final_recommendation_product_index.csv"
)

FINAL_METADATA_PATH = (
    MODEL_DIR
    / "final_recommendation_metadata.json"
)

FINAL_EVALUATION_PATH = (
    REPORT_DIR
    / "final_recommendation_evaluation.csv"
)

HYBRID_TUNING_PATH = (
    REPORT_DIR
    / "hybrid_weight_tuning.csv"
)


# Save the selected leakage-free TF-IDF artifacts.
joblib.dump(
    evaluation_tfidf_vectorizer,
    FINAL_VECTORIZER_PATH,
)

save_npz(
    FINAL_MATRIX_PATH,
    evaluation_tfidf_matrix.astype(
        "float32"
    ),
)


# Preserve the exact product order used by the matrix.
application_columns = [
    column
    for column in [
        "asin",
        "title",
        "description",
        "brand",
        "keyword_list",
        "price",
        "avg_rating",
        "rating_count",
        "image_url",
        "product_url",
    ]
    if column in products.columns
]

final_product_index = products[
    application_columns
].copy()

final_product_index.to_csv(
    FINAL_PRODUCT_INDEX_PATH,
    index=False,
    encoding="utf-8-sig",
)


# Save experiment reports.
final_evaluation_results.to_csv(
    FINAL_EVALUATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

hybrid_tuning_results.to_csv(
    HYBRID_TUNING_PATH,
    index=False,
    encoding="utf-8-sig",
)


# Convert final result values into JSON-compatible types.
selected_result = (
    final_evaluation_results
    .iloc[0]
    .to_dict()
)

selected_result = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value
    in selected_result.items()
}


final_metadata = {
    "selected_method": (
        selected_recommendation_method
    ),
    "model_type": (
        "content_based_tfidf"
    ),
    "product_count": len(products),
    "feature_count": int(
        evaluation_tfidf_matrix.shape[1]
    ),
    "text_fields": [
        "title",
        "brand",
        "description",
    ],
    "keyword_used_as_model_input": False,
    "human_labels_used": False,
    "relevance_definition": (
        "Products sharing at least one "
        "collection keyword"
    ),
    "evaluation_top_k": EVALUATION_TOP_K,
    "tuning_queries": len(
        tuning_query_indices
    ),
    "final_holdout_queries": len(
        final_query_indices
    ),
    "selection_formula": {
        "ndcg@5": 0.60,
        "precision@5": 0.25,
        "catalog_coverage": 0.15,
    },
    "final_metrics": selected_result,
    "deterministic": True,
    "limitation": (
        "Collection keywords are weak relevance "
        "labels rather than customer relevance data."
    ),
}

with open(
    FINAL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_metadata,
        file,
        indent=2,
    )


print("Final recommendation artifacts saved.")
print("Vectorizer:", FINAL_VECTORIZER_PATH)
print("TF-IDF matrix:", FINAL_MATRIX_PATH)
print("Product index:", FINAL_PRODUCT_INDEX_PATH)
print("Metadata:", FINAL_METADATA_PATH)
print("Evaluation report:", FINAL_EVALUATION_PATH)
print("Hybrid tuning report:", HYBRID_TUNING_PATH)

Final recommendation artifacts saved.
Vectorizer: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/models/recommendation/final_tfidf_vectorizer.joblib
TF-IDF matrix: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/models/recommendation/final_tfidf_matrix.npz
Product index: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/models/recommendation/final_recommendation_product_index.csv
Metadata: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/models/recommendation/final_recommendation_metadata.json
Evaluation report: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/recommendation/final_recommendation_evaluation.csv
Hybrid tuning report: /content/drive/MyDrive/Amazon-Marketplace-Product-Intelligence-Platform-BJIT/reports/recommendation/hybrid_weight_tuning.csv
